In [ ]:
import os
import sys

# --- FORCED SYSTEM FIXES FOR SILENT CRASHES ---
# Allow duplicate OpenMP runtime initializations (PyTorch vs libvips)
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
# Force single threading on core utilities to isolate library collisions
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

print("[CHECKPOINT 1/8] Setting environment flags done. Importing core libraries...")
import json
import cv2
import numpy as np
import pyvips
from cellpose import models
from shapely.geometry import Polygon, box
print("[CHECKPOINT 2/8] Core libraries imported successfully.")

# ==========================================
# 1. CONFIGURATION & HYPERPARAMETERS
# ==========================================
IMAGE_PATH = "/Users/nadavyayon/Downloads/20260424_wga_test1_wellview_c2_afterstain_011B2.vsi"
OUTPUT_GEOJSON = "/Users/nadavyayon/Downloads/20260424_wga_test1_wellview_c2_afterstain_011B2_qupath_stitched_cells.geojson"

TILE_SIZE = 2048   
MARGIN = 256       
STRIDE = TILE_SIZE - (2 * MARGIN)  
CHANNELS = [0, 0]  # [Cyto, Nuclei]. 0 = none. Update after running preview cell if needed.
EXPECTED_DIAMETER_MICRONS = 30.0  # The real-world physical cell size
PIXEL_SIZE_MICRONS = 0.25         # Get this from metadata (e.g., slideio resolution)

# Calculate derived diameter
DIAMETER_PX = EXPECTED_DIAMETER_MICRONS / PIXEL_SIZE_MICRONS

# ==========================================
# 2. INITIALIZATION
# ==========================================
print(f"[CHECKPOINT 3/8] Opening image stream for VSI file: {IMAGE_PATH}")
try:
    image = pyvips.Image.new_from_file(IMAGE_PATH)
    img_width, img_height = image.width, image.height
    print(f"-> SUCCESS: Slide Dimensions: {img_width}x{img_height}, Bands/Channels: {image.bands}")
except Exception as e:
    print(f"-> FATAL CRASH at pyvips file reading: {e}")
    sys.exit(1)

print("[CHECKPOINT 4/8] Initializing Cellpose 4 model (`cpdino` on CPU)...")
try:
    model_cells = models.CellposeModel(gpu=False, model_type='cpdino')
    print("-> SUCCESS: CellposeModel initialized.")
except Exception as e:
    print(f"-> FATAL CRASH at Cellpose model initialization: {e}")
    sys.exit(1)

all_global_features = []
cell_counter = 0

# ==========================================
# 3. THE TILING & PROCESSING LOOP
# ==========================================
print("[CHECKPOINT 5/8] Entering the sliding-window processing loop...")

for y in range(0, img_height, STRIDE):
    for x in range(0, img_width, STRIDE):
        
        w = min(TILE_SIZE, img_width - x)
        h = min(TILE_SIZE, img_height - y)
        
        # Adjust margin check: if the total dimension is smaller than 2*MARGIN, process without margin filter
        # but warn that stitching might be imperfect.
        current_margin_x = MARGIN if w > 2 * MARGIN else 0
        current_margin_y = MARGIN if h > 2 * MARGIN else 0
        
        if (w <= 0 or h <= 0):
            continue
            
        print(f"\n--- [TILE START] Processing Tile at X: {x}, Y: {y} | Window size: {w}x{h} ---")
        
        print("   [Step A] Cropping tile via pyvips...")
        tile = image.crop(x, y, w, h)
        
        print("   [Step B] Calling write_to_memory() and constructing NumPy array...")
        try:
            # Memory allocations here can crash if the image size/bands configuration is unexpected
            img_np = np.ndarray(
                buffer=tile.write_to_memory(),
                dtype=np.uint8,
                shape=[tile.height, tile.width, tile.bands]
            )
            print(f"   -> NumPy shape achieved: {img_np.shape}")
        except Exception as e:
            print(f"   -> FATAL CRASH during memory conversion: {e}")
            sys.exit(1)
        
        print(f"   [Step C] Sending image array to `model_cells.eval()` (Diameter: {DIAMETER_PX:.2f} px)...")
        try:
            masks, flows, styles = model_cells.eval(img_np, channels=CHANNELS, diameter=DIAMETER_PX if DIAMETER_PX > 0 else None)
            print(f"   -> SUCCESS: Inference completed. Found {len(np.unique(masks)) - 1} raw masks.")
        except Exception as e:
            print(f"   -> FATAL CRASH inside Cellpose model evaluation: {e}")
            sys.exit(1)
        
        print("   [Step D] Filtering borders and processing contours via OpenCV/Shapely...")
        # Use dynamic margins to avoid empty boxes on small images
        local_valid_box = box(current_margin_x, current_margin_y, w - current_margin_x, h - current_margin_y)
        unique_ids = np.unique(masks)
        unique_ids = unique_ids[unique_ids != 0] 
        
        tile_added_cells = 0
        for obj_id in unique_ids:
            obj_mask = (masks == obj_id).astype(np.uint8) * 255
            contours, _ = cv2.findContours(obj_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            
            for contour in contours:
                if len(contour) < 3:
                    continue  
                
                poly_points = contour.reshape(-1, 2).tolist()
                local_poly = Polygon(poly_points)
                
                if local_poly.area < 15:
                    continue 
                    
                # Include object if its centroid is within the valid inner tile area
                if not local_valid_box.contains(local_poly.centroid):
                    continue
                
                local_poly = local_poly.simplify(0.4, preserve_topology=True)
                if local_poly.is_empty or not isinstance(local_poly, Polygon):
                    continue
                
                simplified_points = list(local_poly.exterior.coords)
                global_points = [[pt[0] + x, pt[1] + y] for pt in simplified_points]
                global_points.append(global_points[0]) 
                
                cell_counter += 1
                tile_added_cells += 1
                feature = {
                    "type": "Feature",
                    "id": f"cell_{cell_counter}",
                    "geometry": {
                        "type": "Polygon",
                        "coordinates": [global_points]
                    },
                    "properties": {
                        "objectType": "Cell",
                        "classification": {"name": "Cell"}
                    }
                }
                all_global_features.append(feature)
        
        print(f"   [Step E] Finished Tile. Saved {tile_added_cells} valid objects from this block.")
        
        # --- DEBUG BREAK ---
        # Stop script after the very first tile to see if everything up to here works fine.
        print("\n[DEBUG BREAK] First tile passed flawlessly. Stopping execution here for safety.")
        sys.exit(0)

# ==========================================
# 4. EXPORT TO GEOJSON
# ==========================================
print(f"\n[CHECKPOINT 6/8] Segmentation Complete! Total cells found: {cell_counter}")
geojson_data = {"type": "FeatureCollection", "features": all_global_features}

print(f"[CHECKPOINT 7/8] Writing data to disk: {OUTPUT_GEOJSON}")
with open(OUTPUT_GEOJSON, 'w') as f:
    json.dump(geojson_data, f)

print("[CHECKPOINT 8/8] Pipeline finished completely and successfully!")

[CHECKPOINT 1/8] Setting environment flags done. Importing core libraries...
[CHECKPOINT 2/8] Core libraries imported successfully.
[CHECKPOINT 3/8] Opening image stream for VSI file: /Users/nadavyayon/Downloads/20260424_wga_test1_wellview_c2_afterstain_011B2.vsi
-> SUCCESS: Slide Dimensions: 512x512, Bands/Channels: 3
[CHECKPOINT 4/8] Initializing Cellpose 4 model (`cpdino` on CPU)...
-> SUCCESS: CellposeModel initialized.
[CHECKPOINT 5/8] Entering the sliding-window processing loop...

--- [TILE START] Processing Tile at X: 0, Y: 0 | Window size: 512x512 ---
   [Step A] Cropping tile via pyvips...
   [Step B] Calling write_to_memory() and constructing NumPy array...
   -> NumPy shape achieved: (512, 512, 3)
   [Step C] Sending image array to `model_cells.eval()` (Cellpose Inference)...
   -> SUCCESS: Inference completed. Found 122 raw masks.
   [Step D] Filtering borders and processing contours via OpenCV/Shapely...
   [Step E] Finished Tile. Saved 0 valid objects from this block.

[

/Users/nadavyayon/mamba/envs/cellpose/lib/python3.12/site-packages/cellpose/dynamics.py:541: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/Context.cpp:767.)
  coo = torch.sparse_coo_tensor(pt, torch.ones(pt.shape[1], device=pt.device, dtype=torch.int),


SystemExit: 0

/Users/nadavyayon/mamba/envs/cellpose/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3756: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


[PREVIEW] Opening image stream for VSI file: /Users/nadavyayon/Downloads/20260424_wga_test1_wellview_c2_afterstain_011B2.vsi[page=0]
-> WARNING: 'ome-xml' field missing. Using numerical fallbacks.

      AUTOMATED METADATA VERIFICATION REPORT
Slide Dimensions  : 512x512
Detected Channels : CHANNEL_0, CHANNEL_1, CHANNEL_2
Selected Cyto     : 'CY3' -> Index 0
Selected Nucleus  : 'DAPI' -> Index 0
Cellpose CHANNELS : [0, 0]
Extracted Scale   : 264.5833 µm/pixel
Model Diameter    : 0.11 pixels

Generating low-res thumbnail for visual inspection...
-> SUCCESS: Thumbnail saved to './slide_preview.png'. Check it to verify channels.


In [ ]:
import os
import sys
import slideio
import numpy as np
import cv2

# ==========================================
# 1. USER CONFIGURATION
# ==========================================
IMAGE_PATH = "/Users/nadavyayon/Downloads/20260424_wga_test1_wellview_c2_afterstain_011B2.vsi"

CYTOPLASM_CHANNEL_NAME = "CY3"  
NUCLEUS_CHANNEL_NAME = "DAPI"
EXPECTED_DIAMETER_MICRONS = 30.0  

# ==========================================
# 2. FILE OPENING & SCENE SELECTION
# ==========================================
print(f"[PREVIEW] Opening VSI bundle via slideio: {IMAGE_PATH}")

try:
    slide = slideio.open_slide(IMAGE_PATH, "VSI")
except Exception as e:
    print(f"-> FATAL ERROR: Could not open VSI file. ({e})")
    sys.exit(1)

target_scene = None
max_pixels = 0

print("Scanning internal VSI scenes...")
for i in range(slide.num_scenes):
    scene = slide.get_scene(i)
    
    # FIX: Extract dimensions using the .size property tuple (width, height)
    width, height = scene.size
    pixels = width * height
    print(f" -> Found Scene [{i}]: Dimensions = {width}x{height} ({scene.num_channels} channels) | Name: {scene.name}")
    
    # Auto-select the massive high-res layer
    if pixels > max_pixels:
        max_pixels = pixels
        target_scene = scene

if target_scene is None:
    print("-> FATAL ERROR: No valid image scenes found.")
    sys.exit(1)

# Grab the final verified dimensions
img_width, img_height = target_scene.size
total_bands = target_scene.num_channels
print(f"\n[SUCCESS] Locked onto Scene: '{target_scene.name}' ({img_width}x{img_height})")

# ==========================================
# 3. METADATA EXTRACTION 
# ==========================================
# Extract channel names natively
channel_names = [target_scene.get_channel_name(i).upper() for i in range(total_bands)]

# Map Strings to Cellpose 1-Based Indices
cellpose_cyto_idx = 0
cellpose_nucl_idx = 0
for i, name in enumerate(channel_names):
    if CYTOPLASM_CHANNEL_NAME.upper() in name:
        cellpose_cyto_idx = i + 1 
    if NUCLEUS_CHANNEL_NAME.upper() in name:
        cellpose_nucl_idx = i + 1

CHANNELS = [cellpose_cyto_idx, cellpose_nucl_idx]

# Extract Physical Resolution (slideio returns meters per pixel)
resolution_tuple = target_scene.resolution 
if resolution_tuple is not None and resolution_tuple[0] > 0:
    PIXEL_SIZE_MICRONS = resolution_tuple[0] * 1_000_000 
else:
    PIXEL_SIZE_MICRONS = 0.1621

DIAMETER_PX = EXPECTED_DIAMETER_MICRONS / PIXEL_SIZE_MICRONS

print("\n" + "="*50)
print("      AUTOMATED METADATA VERIFICATION REPORT")
print("="*50)
print(f"Slide Dimensions     : {img_width}x{img_height}")
print(f"Detected Channels    : {', '.join(channel_names)}")
print(f"Selected Cyto        : '{CYTOPLASM_CHANNEL_NAME}' -> Index {cellpose_cyto_idx}")
print(f"Selected Nucleus     : '{NUCLEUS_CHANNEL_NAME}' -> Index {cellpose_nucl_idx}")
print(f"Cellpose CHANNELS    : {CHANNELS}")
print(f"Extracted Scale      : {PIXEL_SIZE_MICRONS:.4f} µm/pixel")
print(f"Model Diameter       : {DIAMETER_PX:.2f} pixels")
print("="*50 + "\n")

# ==========================================
# 4. LOW-RES THUMBNAIL VISUALIZATION
# ==========================================
print("Generating low-res thumbnail from master resolution...")
thumb_width = 1000
thumb_height = int(img_height * (thumb_width / img_width))

# Read downsampled block using slideio
thumb_np = target_scene.read_block((0, 0, img_width, img_height), size=(thumb_width, thumb_height))

# slideio returns arrays formatted as (Channels, Height, Width). 
# Transpose it to standard standard Image format (Height, Width, Channels)
if len(thumb_np.shape) == 3:
    thumb_np = thumb_np.transpose(1, 2, 0)
else:
    thumb_np = np.expand_dims(thumb_np, axis=2)

# Convert 16-bit data to 8-bit for quick preview rendering
if thumb_np.dtype == np.uint16:
    thumb_np = (thumb_np / 256).astype(np.uint8)

preview_img = np.zeros((thumb_height, thumb_width, 3), dtype=np.uint8)
if cellpose_cyto_idx > 0 and cellpose_cyto_idx <= thumb_np.shape[2]:
    preview_img[:, :, 2] = thumb_np[:, :, cellpose_cyto_idx - 1] # Red
if cellpose_nucl_idx > 0 and cellpose_nucl_idx <= thumb_np.shape[2]:
    preview_img[:, :, 0] = thumb_np[:, :, cellpose_nucl_idx - 1] # Blue

cv2.imwrite("./slide_preview.png", preview_img)
print("-> SUCCESS: Thumbnail saved to './slide_preview.png'. Everything is configured.")

[PREVIEW] Opening VSI bundle via slideio: /Users/nadavyayon/Downloads/20260424_wga_test1_wellview_c2_afterstain_011B2.vsi
Scanning internal VSI scenes...


AttributeError: 'Scene' object has no attribute 'width'